In [1]:
# pip install arxiv
import importlib
import json

arxiv = importlib.import_module("arxiv")

client = arxiv.Client(page_size=200, delay_seconds=3, num_retries=5)
queries = [
    'abs:"retrieval augmented generation"',
    'abs:"dense retrieval"',
    'abs:"semantic search"',
    'abs:"large language model" AND abs:retrieval',
]

docs = {}
for q in queries:
    search = arxiv.Search(
        query=f'({q}) AND (cat:cs.CL OR cat:cs.IR OR cat:cs.LG)',
        max_results=3000,
        sort_by=arxiv.SortCriterion.Relevance,
    )
    for r in client.results(search):
        docs[r.entry_id] = {
            "id": r.entry_id,
            "title": r.title,
            "abstract": r.summary.replace("\n", " "),
            "year": r.published.year,
        }

with open("corpus.jsonl", "w") as f:
    for d in docs.values():
        f.write(json.dumps(d) + "\n")
        
print(len(docs))

5648


In [2]:
import pandas as pd

df = pd.read_json("corpus.jsonl", lines=True)
df["n_words"] = df["abstract"].str.split().str.len()

print(df.shape)
print(df["n_words"].describe())
print(df["year"].value_counts().sort_index())
print("empty abstracts:", (df["abstract"].str.strip() == "").sum())
print("duplicate titles:", df["title"].str.lower().duplicated().sum())

print(df.sample(3, random_state=0)[["title", "abstract"]].to_string())

(5648, 5)
count    5648.000000
mean      169.970786
std        38.292621
min        36.000000
25%       144.000000
50%       168.000000
75%       193.000000
max       313.000000
Name: n_words, dtype: float64
year
2009       1
2010       1
2011       1
2012       6
2013       5
2014       6
2015       1
2016       5
2017       7
2018      17
2019      11
2020      20
2021      73
2022     133
2023     386
2024    1323
2025    2089
2026    1563
Name: count, dtype: int64
empty abstracts: 0
duplicate titles: 2
                                                                                                                        title                                                                                                                                                                                                                                                                                                                                                                           

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_json("corpus.jsonl", lines=True)
df = df.drop_duplicates(subset="title", keep="first").reset_index(drop=True)
df["text"] = df["title"] + ". " + df["abstract"]   # title helps retrieval
df.to_json("corpus_clean.jsonl", orient="records", lines=True)

print(df.nsmallest(3, "n_words")[["title", "abstract"]] if "n_words" in df else
      df.assign(n=df.abstract.str.split().str.len()).nsmallest(3, "n")[["title", "abstract"]])

# Baseline config: decide now, then freeze it
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True,
                      ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(df["text"])
print(X.shape)

def search_tfidf(query, k=5):
    q = vec.transform([query])
    scores = cosine_similarity(q, X).ravel()
    top = scores.argsort()[::-1][:k]
    return [(df.title[i], round(float(scores[i]), 3)) for i in top]

for q in ["how do models look things up instead of memorizing facts",
          "reducing made-up answers from language models"]:
    print("\n", q)
    for t, s in search_tfidf(q):
        print(f"  {s}  {t}")

                                                  title  \
30    Towards Comprehensive Vietnamese Retrieval-Aug...   
15    Observations on Building RAG Systems for Techn...   
3127  Dense Retrieval for Low Resource Languages -- ...   

                                               abstract  
30    This paper presents our contributions towards ...  
15    Retrieval augmented generation (RAG) for techn...  
3127  This paper reports some difficulties and some ...  
(5646, 81521)

 how do models look things up instead of memorizing facts
  0.102  Generative AI for Low-Carbon Artificial Intelligence of Things with Large Language Models
  0.098  Generative Dense Retrieval: Memory Can Be a Burden
  0.094  Provable Benefits of In-Tool Learning for Large Language Models
  0.082  Generative Cross-Modal Retrieval: Memorizing Images in Multimodal Language Models for Retrieval and Beyond
  0.068  Semantic Advertising

 reducing made-up answers from language models
  0.105  FilterRAG: Zero-Shot In

In [12]:
import json

queries = {
    "q1": "how do models look things up instead of memorizing facts",
    "q2": "reducing made-up answers from language models",
    "q3": "splitting long documents into pieces before indexing",
    "q4": "reordering retrieved results with a second model",
    "q5": "question answering over tables and scanned documents",
    "q6": "retrieval for languages with little training data",
    "q7": "measuring whether a generated answer is supported by the sources",
    "q8": "making retrieval faster and cheaper at scale",
    # add your own up to ~30-40
}
json.dump(queries, open("queries.json", "w"), indent=2)

In [13]:
import os
print(os.path.exists("queries.json"))   # should print True

True


In [16]:
import json, os
import numpy as np

LABELS = "labels.json"   # {qid: {doc_idx: 0/1/2}}
labels = json.load(open(LABELS)) if os.path.exists(LABELS) else {}

if os.path.exists("queries.json"):
    queries = json.load(open("queries.json"))
else:
    queries = {
        "q1": "how do models look things up instead of memorizing facts",
        "q2": "reducing made-up answers from language models",
    }
    print("queries.json not found; using built-in evaluation queries.")

def label_pool(system_fn, k=10):
    """system_fn(query, k) -> list of doc indices. Judge unseen docs only."""
    for qid, q in queries.items():
        seen = labels.setdefault(qid, {})
        for i in system_fn(q, k):
            if str(i) in seen:
                continue
            print(f"\nQ: {q}\n[{i}] {df.title[i]}\n{df.abstract[i][:500]}")
            seen[str(i)] = int(input("0=no 1=partial 2=yes > "))
            json.dump(labels, open(LABELS, "w"))

def mrr_ndcg(system_fn, k=10):
    rr, nd = [], []
    for qid, q in queries.items():
        rel = labels.get(qid, {})
        ranked = system_fn(q, k)
        gains = [rel.get(str(i), 0) for i in ranked]
        first = next((r for r, g in enumerate(gains, 1) if g >= 1), None)
        rr.append(1 / first if first else 0)
        dcg = sum((2**g - 1) / np.log2(r + 1) for r, g in enumerate(gains, 1))
        ideal = sorted(rel.values(), reverse=True)[:k]
        idcg = sum((2**g - 1) / np.log2(r + 1) for r, g in enumerate(ideal, 1))
        nd.append(dcg / idcg if idcg else 0)
    return np.mean(rr), np.mean(nd)

# TF-IDF returning indices
def tfidf_idx(q, k=10):
    s = cosine_similarity(vec.transform([q]), X).ravel()
    return s.argsort()[::-1][:k].tolist()

queries.json not found; using built-in evaluation queries.
